Importo le librerie per poter estrarre 500 frasi randomiche dal corpus

In [ ]:
with open('en-it.tmx.txt.en') as file:
    text_en = file.read()

#print(text_en)

In [ ]:
with open('en-it.tmx.txt.it') as file:
    text_it = file.read()

#print(text_it)

In [ ]:
import nltk
import pandas as pd
import torch
from nltk.tokenize import sent_tokenize, word_tokenize
from sentence_transformers import SentenceTransformer, util

# Download delle risorse NLTK necessarie
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

def elabora_corpus_bilingue(testo_en, testo_it, min_parole=6, max_parole=15, target_coppie=500, soglia_confidenza=0.75):

    # Funzione per tokenizzare e filtrare per lunghezza (minima e massima)
    def filtra_frasi(testo, lingua):
        frasi = sent_tokenize(testo, language=lingua)
        frasi_valide = []
        for frase in frasi:
            parole = [p for p in word_tokenize(frase, language=lingua) if p.isalnum()]
            # Nuova condizione: verifica che la lunghezza sia compresa tra min e max
            if min_parole <= len(parole) <= max_parole:
                frasi_valide.append(frase)
        return frasi_valide

    print("1. Estrazione e filtraggio delle frasi in corso...")
    frasi_en = filtra_frasi(testo_en, 'english')
    frasi_it = filtra_frasi(testo_it, 'italian')

    print(f"Frasi valide trovate: {len(frasi_en)} (EN), {len(frasi_it)} (IT)")

    # Inizializzazione del modello vettoriale
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"2. Caricamento del modello vettoriale su: {device.upper()}")
    modello = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=device)

    # Generazione degli embedding
    print("3. Calcolo delle rappresentazioni semantiche e allineamento...")
    emb_en = modello.encode(frasi_en, convert_to_tensor=True)
    emb_it = modello.encode(frasi_it, convert_to_tensor=True)

    # Calcolo similarità
    matrice_similarita = util.cos_sim(emb_en, emb_it)

    coppie_estratte = []
    indici_it_assegnati = set()

    for i in range(len(frasi_en)):
        if len(coppie_estratte) >= target_coppie:
            break

        miglior_idx_it = matrice_similarita[i].argmax().item()
        punteggio = matrice_similarita[i][miglior_idx_it].item()

        if punteggio >= soglia_confidenza and miglior_idx_it not in indici_it_assegnati:
            coppie_estratte.append({
                "Inglese": frasi_en[i],
                "Italiano": frasi_it[miglior_idx_it],
                "Affidabilità": round(punteggio, 3)
            })
            indici_it_assegnati.add(miglior_idx_it)

    return pd.DataFrame(coppie_estratte)

# --- AVVIO DELL'ELABORAZIONE ---
# Passaggio dei nuovi parametri min_parole e max_parole
df_allineato = elabora_corpus_bilingue(text_en, text_it, min_parole=6, max_parole=15, target_coppie=500)

# Visualizzazione dei risultati
print(f"\nElaborazione completata. Totale coppie estratte: {len(df_allineato)}")
display(df_allineato.head())

# Salvataggio in formato CSV
df_allineato.to_csv('frasi_allineate_6_15.csv', index=False, encoding='utf-8')
print("\nIl file 'frasi_allineate_6_15.csv' è stato salvato nei file di Colab.")

1. Estrazione e filtraggio delle frasi in corso...
Frasi valide trovate: 13540 (EN), 12612 (IT)
2. Caricamento del modello vettoriale su: CPU


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

3. Calcolo delle rappresentazioni semantiche e allineamento...

Elaborazione completata. Totale coppie estratte: 500


,Inglese,Italiano,Affidabilità
0,The color and opacity of the highlight border.,I rettangolini nero e bianco reimpostano i col...,0.799
1,Accerciser is an interactive Python accessibil...,Accerciser è un esploratore interattivo di acc...,0.944
2,"In essence, Accerciser is a next generation at...","Essenzialmente, Accerciser è uno strumento at-...",0.859
3,Accerciser Main Window\nShows main window.,Finestra principale di Accerciser\nVisualizza ...,0.923
4,"The menu bar contains File, Edit, Bookmarks, V...","Un elenco corredato di informazioni sull'uso, ...",0.762



Il file 'frasi_allineate_6_15.csv' è stato salvato nei file di Colab.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import csv

print("--- INIZIO ELABORAZIONE E PARTIZIONE DEL DATASET ---")

# 1. Caricamento del dataset con gestione avanzata degli errori di parsing testuale
# - engine='python': utilizza un motore più flessibile per la lettura
# - on_bad_lines='skip': ignora eventuali righe malformate (es. virgole extra nel testo)
# - quoting=csv.QUOTE_MINIMAL: ottimizza la gestione delle virgolette interne
df_completo = pd.read_csv(
    'frasi_allineate_6_15.csv',
    engine='python',
    on_bad_lines='skip',
    quoting=csv.QUOTE_MINIMAL
)

print(f"Dataset caricato. Totale coppie valide al netto di errori di formattazione: {len(df_completo)}")

# 2. Divisione in Training Set (80%) e Test Set (20%)
# Il parametro random_state=42 fissa il "seme" di casualità, garantendo la totale riproducibilità
df_train, df_test = train_test_split(df_completo, test_size=0.20, random_state=42)

print(f"Dimensione Training Set: {len(df_train)} coppie")
print(f"Dimensione Test Set: {len(df_test)} coppie")

# 3. ESPORTAZIONE BILINGUE (Per la fase di Fine-Tuning dei modelli)
# Si mantiene la struttura tabellare originale
df_train.to_csv('train_bilingue.csv', index=False, encoding='utf-8')
df_test.to_csv('test_bilingue.csv', index=False, encoding='utf-8')
print("- File bilingui (.csv) salvati.")

# 4. ESPORTAZIONE MONOLINGUA (Per la fase di Inferenza e la creazione del Gold Standard)
# Impostiamo index=False e header=False per esportare file di testo puro (una frase per riga)

# A. Inglese (Segmenti sorgente da fornire ai modelli durante il test)
df_train['Inglese'].to_csv('train_en.txt', index=False, header=False, encoding='utf-8')
df_test['Inglese'].to_csv('test_en.txt', index=False, header=False, encoding='utf-8')
print("- File monolingua in Inglese (.txt) salvati.")

# B. Italiano (Gold Standard per il calcolo finale delle metriche di valutazione)
df_train['Italiano'].to_csv('train_it.txt', index=False, header=False, encoding='utf-8')
df_test['Italiano'].to_csv('test_it.txt', index=False, header=False, encoding='utf-8')
print("- File monolingua in Italiano (.txt) salvati.")

print("\n--- OPERAZIONE COMPLETATA CON SUCCESSO ---")

--- INIZIO ELABORAZIONE E PARTIZIONE DEL DATASET ---
Dataset caricato. Totale coppie valide al netto di errori di formattazione: 303
Dimensione Training Set: 242 coppie
Dimensione Test Set: 61 coppie
- File bilingui (.csv) salvati.
- File monolingua in Inglese (.txt) salvati.
- File monolingua in Italiano (.txt) salvati.

--- OPERAZIONE COMPLETATA CON SUCCESSO ---
